In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

from relbench.base import BaseTask, Dataset
from relbench.datasets import get_dataset
from relbench.tasks import get_task, get_task_names

In [3]:
dataset_f1 = get_dataset(name="rel-f1", download=True)
db_f1 = dataset_f1.get_db()

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.04 seconds.


In [4]:
from rtgl.path_builder import PathBuilder

In [5]:
path_builder = PathBuilder(db_f1, {})

In [ ]:
# cte_dict = {
#     "cte1": (Table(
#         df=None,
#         fkey_col_to_pkey_table={"col1": "races", "col2": "constructors"},
#         pkey_col="id",
#         time_col=None
#     ), {"results": "resulttttttt"})
# }

In [5]:
from rtgl.base import build_paths_from_table, build_paths, extend_relations

In [11]:
extend_relations(db_f1, cte_dict)

({'drivers': {},
  'races': {'circuits': ('circuitId', 'f')},
  'constructor_standings': {'races': ('raceId', 'f'),
   'constructors': ('constructorId', 'f')},
  'standings': {'races': ('raceId', 'f'), 'drivers': ('driverId', 'f')},
  'qualifying': {'races': ('raceId', 'f'),
   'drivers': ('driverId', 'f'),
   'constructors': ('constructorId', 'f')},
  'circuits': {},
  'constructor_results': {'races': ('raceId', 'f'),
   'constructors': ('constructorId', 'f')},
  'results': {'races': ('raceId', 'f'),
   'drivers': ('driverId', 'f'),
   'constructors': ('constructorId', 'f'),
   'cte1': ('resulttttttt', 'f')},
  'constructors': {},
  'cte1': {'races': ('col1', 'f'), 'constructors': ('col2', 'f')}},
 {'drivers': {'standings': ('driverId', 'r'),
   'qualifying': ('driverId', 'r'),
   'results': ('driverId', 'r')},
  'races': {'circuits': ('circuitId', 'f'),
   'constructor_standings': ('raceId', 'r'),
   'standings': ('raceId', 'r'),
   'qualifying': ('raceId', 'r'),
   'constructor_resu

In [6]:
paths = build_paths(db_f1, {})
paths

{'drivers': {'drivers': [],
  'standings': [('driverId', 'standings', 'r')],
  'qualifying': [('driverId', 'qualifying', 'r')],
  'results': [('driverId', 'results', 'r')],
  'races': [('driverId', 'results', 'r'), ('raceId', 'races', 'f')],
  'constructors': [('driverId', 'results', 'r'),
   ('constructorId', 'constructors', 'f')],
  'circuits': [('driverId', 'results', 'r'),
   ('raceId', 'races', 'f'),
   ('circuitId', 'circuits', 'f')],
  'constructor_standings': [('driverId', 'results', 'r'),
   ('constructorId', 'constructors', 'f'),
   ('constructorId', 'constructor_standings', 'r')],
  'constructor_results': [('driverId', 'results', 'r'),
   ('constructorId', 'constructors', 'f'),
   ('constructorId', 'constructor_results', 'r')]},
 'races': {'races': [],
  'circuits': [('circuitId', 'circuits', 'f')],
  'constructor_standings': [('raceId', 'constructor_standings', 'r')],
  'standings': [('raceId', 'standings', 'r')],
  'qualifying': [('raceId', 'qualifying', 'r')],
  'construc

In [7]:
path_builder.build_path("races", "constructors")


⚠️ [WARNING] UserWarning:
Multiple paths found between 'races' and 'constructors'!
Using the first shortest path: [('raceId', 'constructor_standings', 'r'), ('constructorId', 'constructors', 'f')].


[('raceId', 'constructor_standings', 'r'),
 ('constructorId', 'constructors', 'f')]

In [38]:
db_f1.table_dict["constructors"]

Table(df=
     constructorId constructorRef            name nationality
0                0        mclaren         McLaren     British
1                1     bmw_sauber      BMW Sauber      German
2                2       williams        Williams     British
3                3        renault         Renault      French
4                4     toro_rosso      Toro Rosso     Italian
..             ...            ...             ...         ...
206            206          manor  Manor Marussia     British
207            207           haas    Haas F1 Team    American
208            208   racing_point    Racing Point     British
209            209     alphatauri      AlphaTauri     Italian
210            210         alpine  Alpine F1 Team      French

[211 rows x 4 columns],
  fkey_col_to_pkey_table={},
  pkey_col=constructorId,
  time_col=None)

In [5]:
from rtgl.converter import SConverter

In [6]:
converter_f1 = SConverter(db_f1)
squery = """
    PREDICT SUM(races.round)
    FOR EACH constructors.*
    WHERE constructors.constructorId > 20
      AND constructors.constructorId < 100;
"""

In [7]:
converter_f1.convert(squery, execute=True)
# table.df[table.df["label"] == False]

------------------ Table ------------------
DataFrame:
      fk   label
0      0  5374.0
1      1   648.0
2      2  4791.0
3      3  2411.0
4      4   648.0
..   ...     ...
142  197    66.0
143  198    60.0
144  199   105.0
145  200    39.0
146  201   256.0

[147 rows x 2 columns]
Foreign Key Columns to Primary Key Tables: {'fk': 'constructors'}
Primary Key Column: None
Time Column: None
-------------------------------------------

In [10]:
def get_timestamps(dataset: Dataset, 
                   timedelta: pd.Timedelta, 
                   num_eval_timestamps: int, 
                   split: str) -> "pd.Series[pd.Timestamp]":
    db = dataset.get_db(upto_test_timestamp=(split != "test"))

    if split == "train":
        start = dataset.val_timestamp - timedelta
        end = db.min_timestamp
        freq = -timedelta
    elif split == "val":
        start = dataset.val_timestamp
        end = min(
            dataset.val_timestamp
            + timedelta * (num_eval_timestamps - 1),
            dataset.test_timestamp - timedelta,
            )
        freq = timedelta
    elif split == "test":
        start = dataset.test_timestamp
        end = min(
            dataset.test_timestamp
            + timedelta * (num_eval_timestamps - 1),
            db.max_timestamp - timedelta,
            )
        freq = timedelta
    else:
        pass

    timestamps = pd.date_range(start=start, end=end, freq=freq)
    return timestamps

In [8]:
from rtgl.converter import TConverter

In [11]:
tconverter_f1 = TConverter(db_f1, get_timestamps(dataset_f1, pd.Timedelta(days=30), 10, "train"))
tquery = """
    PREDICT AVG(races.round, 0, 30, DAYS) IS NULL
    FOR EACH constructors.*;
"""

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.03 seconds.


In [12]:
df = tconverter_f1.convert(tquery, execute=True).df

In [15]:
df[df["label"] == True]

,fk,timestamp,label
0,0,1950-05-20,True
1,1,1950-05-20,True
2,2,1950-05-20,True
3,3,1950-05-20,True
4,4,1950-05-20,True
...,...,...,...
140310,206,2004-12-02,True
140311,207,2004-12-02,True
140312,208,2004-12-02,True
140313,209,2004-12-02,True


In [22]:
converter_f1._register_db()
tbl = converter_f1.conn.sql(query).df()
print(tbl)

       driverId       driverRef code forename      surname        dob  \
0           660           claes   \N   Johnny        Claes 1916-08-11   
1           790  leslie_johnson   \N   Leslie      Johnson 1912-03-22   
2           579          fangio   \N     Juan       Fangio 1911-06-24   
3           661    peter_walker   \N    Peter       Walker 1912-10-07   
4           789          martin   \N   Eugène       Martin 1915-03-24   
...         ...             ...  ...      ...          ...        ...   
20318         1        heidfeld  HEI     Nick     Heidfeld 1977-05-10   
20319        21     barrichello  BAR   Rubens  Barrichello 1972-05-23   
20320        17          button  BUT   Jenson       Button 1980-01-19   
20321        16          webber  WEB     Mark       Webber 1976-08-27   
20322         2         rosberg  ROS     Nico      Rosberg 1985-06-27   

      nationality  results_resultId  results_raceId  results_driverId  ...  \
0         Belgian                 0          